# Olive Fly Detection System

This notebook combines the training and classification components for the olive fly detection project.

## Part 1: Training the Neural Network Classifier

In [1]:
import os
import glob
import cv2
import numpy as np
from skimage.measure import label
from sklearn.neural_network import MLPClassifier

In [2]:
# Use the instructor's background extraction strategy
def extract_foreground_mask(img, kernel_size=9):
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.uint8)
    _, img_bw = cv2.threshold(img_gray, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((kernel_size, kernel_size))
    img_bw_cleaned = cv2.morphologyEx(img_bw, cv2.MORPH_CLOSE, kernel)
    labels = label(img_bw_cleaned)
    if len(img_bw_cleaned.flat) == 0 or np.sum(img_bw_cleaned) == 0:
        return np.zeros_like(img_gray)
    label_of_largest_region = np.argmax(np.bincount(labels.flat, weights=img_bw_cleaned.flat))
    return (labels == label_of_largest_region).astype(np.uint8)

In [3]:
def compute_features(image):
    mask = extract_foreground_mask(image)
    area = float(np.sum(mask))
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        aspect_ratio = float(w) / float(h) if h > 0 else 1.0
    else:
        aspect_ratio = 1.0
        
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    saturation_channel = hsv[:, :, 1]
    mean_saturation = float(cv2.mean(saturation_channel, mask=mask)[0])
    
    return np.array([area, aspect_ratio, mean_saturation])

In [4]:
# Train the model
X_list, y_list = [], []

print("Reading data...")
for f in (glob.glob("not_olive_fly/*.JPG") + glob.glob("not_olive_fly/*.jpg")):
    img = cv2.imread(f)
    if img is not None: X_list.append(compute_features(img)); y_list.append(0)
        
for f in (glob.glob("olive_fly/*.JPG") + glob.glob("olive_fly/*.jpg")):
    img = cv2.imread(f)
    if img is not None: X_list.append(compute_features(img)); y_list.append(1)
        
X, y = np.array(X_list), np.array(y_list)

# Feature Standardization Scaling
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_std[X_std == 0] = 1e-8
X_scaled = (X - X_mean) / X_std

# Define a tiny, highly efficient neural net: 3 inputs -> 4 hidden neurons -> 1 output
mlp = MLPClassifier(hidden_layer_sizes=(4,), activation='relu', max_iter=5000, random_state=42)
mlp.fit(X_scaled, y)


print(f"X_MEAN = np.array({list(X_mean)})")
print(f"X_STD  = np.array({list(X_std)})")
print(f"W1 = np.array({mlp.coefs_[0].tolist()})")
print(f"B1 = np.array({mlp.intercepts_[0].tolist()})")
print(f"W2 = np.array({mlp.coefs_[1].tolist()})")
print(f"B2 = np.array({mlp.intercepts_[1].tolist()})")


Reading data...
X_MEAN = np.array([np.float64(3877.458723404255), np.float64(1.1174638977603828), np.float64(149.3449262126882)])
X_STD  = np.array([np.float64(4801.635565207151), np.float64(0.5524351102376486), np.float64(51.88207312731143)])
W1 = np.array([[-1.4787926909184586, 0.9596893415034591, 1.093172432294036, -1.7454627691021827], [-0.26691265456305974, -1.811491088397156, 0.4203204015223413, 0.38288509439103313], [-0.6746690403539322, 0.17595712582269765, -1.535109766765984, 1.3510070511987022]])
B1 = np.array([0.6876098416420647, -0.8068542754680046, 0.7945428630582831, -1.1543043806619804])
W2 = np.array([[-0.814867770461975], [-0.9349578475965044], [-0.8518958607413155], [-1.6780657705355944]])
B2 = np.array([0.24297322103842567])


## Part 2: Olive Fly Classification and Detection

In [5]:
import glob
import cv2
import numpy as np
from skimage.measure import label

In [6]:
MODEL_PARAMS = {
    'X_MEAN': X_mean,
    'X_STD': X_std,
    'W1': mlp.coefs_[0],
    'B1': mlp.intercepts_[0],
    'W2': mlp.coefs_[1],
    'B2': mlp.intercepts_[1]
}

X_MEAN = MODEL_PARAMS['X_MEAN']
X_STD = MODEL_PARAMS['X_STD']

W1 = MODEL_PARAMS['W1']
B1 = MODEL_PARAMS['B1']

W2 = MODEL_PARAMS['W2']
B2 = MODEL_PARAMS['B2']


In [7]:

def extract_foreground_mask(img, kernel_size=9):
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.uint8)
    _, img_bw = cv2.threshold(img_gray, -1, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    kernel = np.ones((kernel_size, kernel_size))
    img_bw_cleaned = cv2.morphologyEx(img_bw, cv2.MORPH_CLOSE, kernel)
    
    labels = label(img_bw_cleaned)
    flat_labels = labels.flat
    flat_bw = img_bw_cleaned.flat
    
    if len(flat_bw) == 0 or np.sum(flat_bw) == 0:
        return np.zeros_like(img_gray)
        
    label_of_largest_region = np.argmax(np.bincount(flat_labels, weights=flat_bw))
    return (labels == label_of_largest_region).astype(np.uint8)


In [8]:

def detect_olive_fly(image) -> bool:

    try:
        # 1. Extract and standardize physical attributes
        raw_features = compute_features(image)
        scaled_features = (raw_features - X_MEAN) / X_STD
        
        # 2. Hidden Layer Propagation: Z1 = A1*W1 + B1
        z1 = np.dot(scaled_features, W1) + B1
        a2 = np.maximum(0, z1)  # ReLU non-linear activation function
        
        # 3. Output Layer Propagation: Z2 = A2*W2 + B2
        z2 = np.dot(a2, W2) + B2
        
        # Sigmoid activation function mapping to classification boundaries
        probability = 1 / (1 + np.exp(-np.clip(z2[0], -500, 500)))
        
        return bool(probability >= 0.29)
        
    except Exception:
        return False

### Automated Evaluation

In [9]:
def evaluate_classifier(directory):
    TP, TN, FP, FN = 0, 0, 0, 0
    
    for filename in glob.glob(str(directory)+"/**/*.JPG", recursive=True):
        img = cv2.imread(filename)
        if "not_olive_fly" in filename:
            olive_fly = False
        elif "olive_fly" in filename:
            olive_fly = True
        else:
            print(f"{filename} not labeled.")
            continue

        detection_result = detect_olive_fly(img)

        if olive_fly and detection_result:
            TP += 1
        elif olive_fly and not detection_result:
            FN += 1
        elif not olive_fly and detection_result:
            FP += 1
        else:
            TN += 1
            
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")
    if (TP + FP) > 0 and (TP + FN) > 0:
        precision = TP / (TP + FP)
        recall = TP / (TP + FN)
        accuracy = (TP + TN) / (TP + TN + FP + FN)
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

In [10]:
print("Evaluating on training data...\n")
evaluate_classifier(".")

Evaluating on training data...

TP: 145, TN: 1854, FP: 181, FN: 170
Accuracy: 0.8506
Precision: 0.4448, Recall: 0.4603


In [11]:
test_image_path = "olive_fly/IMG_0597 1 referencia.JPG"


if os.path.exists(test_image_path):
    test_img = cv2.imread(test_image_path)
    if test_img is not None:
        result = detect_olive_fly(test_img)
        print(f"Image: {test_image_path}")
        print(f"Detection Result: {'Olive Fly Detected' if result else 'No Olive Fly'}")
    else:
        print(f"Could not read image: {test_image_path}")
else:
    print(f"File not found: {test_image_path}")

Image: olive_fly/IMG_0597 1 referencia.JPG
Detection Result: No Olive Fly
